In [ ]:
from plot_utils import *
setup_plot_style()

X_JITTER = 1e-6

by_L = load_toric_code_data()
print("Found L:", sorted(by_L.keys()))

In [ ]:
# Latency 对比折线图：Micro-blossom / Helios / Ours
palette_lat = {
    'mwpm': '#1f77b4',
    'uf': '#ff7f0e',
    'ours': '#2ca02c',
}

L_TARGETS_LAT = [3, 5, 7, 9, 11]
OUT_DIR_LAT = FIGPLOT_DIR
os.makedirs(OUT_DIR_LAT, exist_ok=True)

fig, axes = plt.subplots(1, len(L_TARGETS_LAT), figsize=(22, 4.8), sharey=False)

for i, L in enumerate(L_TARGETS_LAT):
    ax = axes[i]
    data = by_L.get(L)

    ps_keys = sorted(_HELIOS_TABLE.get(L, {}).keys())
    if not ps_keys:
        ax.set_title(f"Code Distance={L} (N/A)")
        ax.axis('off')
        continue

    mwpm_lat = [get_micro_blossom_cycles(L, p) * SCALE_MICRO_BLOSSOM for p in ps_keys]
    helios_lat = [get_helios_cycles(L, p) * SCALE_HELIOS for p in ps_keys]

    ours_lat = None
    if data is not None:
        eff_total = data.get('latency_means', {}).get('efficient_total', [])
        data_ps = data.get('ps', [])
        if isinstance(eff_total, list) and len(eff_total) == len(data_ps):
            ours_lat = [float(v) * SCALE_TOTAL_TO_CYCLES for v in eff_total]
            ps_ours = data_ps
        else:
            ours_lat = None

    if PLOT_AS_SCATTER:
        xj = 1e-6
        ax.scatter([p - xj for p in ps_keys], mwpm_lat, s=SCATTER_SIZE, marker='o',
                   color=palette_lat['mwpm'], edgecolor='black', linewidths=MARKER_EDGE_WIDTH,
                   label='Micro-Blossom (MWPM Hardware)', zorder=3)
        ax.plot(ps_keys, mwpm_lat, linestyle='-', color=palette_lat['mwpm'], linewidth=2.8)

        ax.scatter(ps_keys, helios_lat, s=SCATTER_SIZE, marker='s',
                   color=palette_lat['uf'], edgecolor='black', linewidths=MARKER_EDGE_WIDTH,
                   label='Helios (UF Hardware)', zorder=3)
        ax.plot(ps_keys, helios_lat, linestyle='-', color=palette_lat['uf'], linewidth=2.8)

        if ours_lat is not None:
            ax.scatter([p + xj for p in ps_ours], ours_lat, s=SCATTER_SIZE, marker='^',
                       color=palette_lat['ours'], edgecolor='black', linewidths=MARKER_EDGE_WIDTH,
                       label='Ours (Hardware)', zorder=3)
            ax.plot(ps_ours, ours_lat, linestyle='-', color=palette_lat['ours'], linewidth=2.8)
    else:
        ax.plot(ps_keys, mwpm_lat, marker='o', linestyle='-', color=palette_lat['mwpm'],
                label='Micro-Blossom (MWPM Hardware)')
        ax.plot(ps_keys, helios_lat, marker='s', linestyle='-', color=palette_lat['uf'],
                label='Helios (UF Hardware)')
        if ours_lat is not None:
            ax.plot(ps_ours, ours_lat, marker='^', linestyle='-', color=palette_lat['ours'],
                    label='Ours (Hardware)')

    ax.grid(True, axis='y', linestyle='--', linewidth=1.2, color='#9a9a9a', alpha=0.5)

    desired_ticks = [p for p in ps_keys if p in (0.0005, 0.001, 0.0015)]
    if desired_ticks:
        ax.set_xticks(desired_ticks)
        ax.set_xticklabels([f"{p:.4f}" for p in desired_ticks], rotation=0, ha='center')

    ax.text(0.95, 0.29, f"Code Distance {L}", transform=ax.transAxes,
            ha='right', va='bottom', fontsize=18)

axes[0].set_ylabel('Decoding Latency (μs)', fontsize=20)

handles_dict = {}
for ax in axes:
    h, l = ax.get_legend_handles_labels()
    for handle, label in zip(h, l):
        if label and label not in handles_dict:
            handles_dict[label] = handle

if handles_dict:
    desired_order = ['Micro-Blossom (MWPM Hardware)', 'Helios (UF Hardware)', 'Ours (Hardware)']
    others = [lbl for lbl in handles_dict if lbl not in desired_order]
    ordered_labels = [lbl for lbl in desired_order if lbl in handles_dict] + others
    ordered_handles = [handles_dict[lbl] for lbl in ordered_labels]
    leg = fig.legend(ordered_handles, ordered_labels, loc='upper center',
                     ncol=len(ordered_labels), frameon=True, bbox_to_anchor=(0.5, 1.045))
    frame = leg.get_frame()
    frame.set_edgecolor('black')
    frame.set_linewidth(1.8)
    frame.set_facecolor('white')
    frame.set_alpha(1.0)

fig.supxlabel('Physical Error Rate', fontsize=24, y=0.06)
fig.tight_layout(rect=[0, 0.002, 1, 0.98])

save_path = os.path.join(OUT_DIR_LAT, 'combined_latency_comparison_row.pdf')
fig.savefig(save_path, dpi=300, bbox_inches='tight', pad_inches=0.02, format='pdf')
plt.show()
print(f"Saved: {save_path}")
try:
    st = os.stat(save_path)
    print(f"File exists, size={st.st_size} bytes")
except FileNotFoundError:
    print("Save failed: file not found.")